<a href="https://colab.research.google.com/github/Pmskabir1234/Torch/blob/main/nnModule.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install opendatasets

In [3]:
import opendatasets as od
df = od.download('https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: saad123
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/yasserh/breast-cancer-dataset


100%|██████████| 48.6k/48.6k [00:00<00:00, 57.4MB/s]

In [28]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
import torch.nn as nn

In [5]:
df = pd.read_csv('/content/breast-cancer-dataset/breast-cancer.csv')
df = df.drop(columns='id')
df.shape

(569, 31)

In [ ]:
df.head(5)

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [6]:
X = df.iloc[:,1:]


In [10]:
df['diagnosis'] = (df['diagnosis']=='M').astype(int)

In [13]:
y = df['diagnosis']

In [14]:
ros = RandomOverSampler()
X,y = ros.fit_resample(X,y)

In [15]:
y.value_counts()

,count
diagnosis,
1,357
0,357


In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, stratify=y)

In [17]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

In [18]:
X_test = scaler.transform(X_test)

In [19]:
y_test = y_test.to_numpy()
y_train = y_train.to_numpy()

In [49]:
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {dev}')

Using device: cuda


In [50]:
X_tr_train = torch.tensor(X_train, dtype=torch.float64).to(dev)
X_tr_test = torch.tensor(X_test, dtype=torch.float64).to(dev)
y_tr_train = torch.tensor(y_train, dtype=torch.float64).to(dev)
y_tr_test = torch.tensor(y_test, dtype=torch.float64).to(dev)

In [51]:
X_tr_train.shape

torch.Size([571, 30])

In [52]:
class NNModel(nn.Module):
  def __init__(self, input_shape):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(input_shape, 3),
        nn.ReLU(),
        nn.Linear(3, 1),
        nn.Sigmoid() # Corrected from nn.sigmoid() to nn.Sigmoid()
    )

  def forward(self,X):
    return self.network(X)



In [53]:
input_shape = X_tr_train.shape[1]

In [54]:
model = NNModel(input_shape)

In [55]:
print(model)

NNModel(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=3, bias=True)
    (1): ReLU()
    (2): Linear(in_features=3, out_features=1, bias=True)
    (3): Sigmoid()
  )
)


In [56]:
!pip install torchinfo

In [60]:
from torchinfo import summary


model = model.to(dev) # Move model to the selected device
summary(model, input_size=(1, input_shape), dtypes=[torch.float64])

Layer (type:depth-idx)                   Output Shape              Param #
NNModel                                  [1, 1]                    --
├─Sequential: 1-1                        [1, 1]                    --
│    └─Linear: 2-1                       [1, 3]                    93
│    └─ReLU: 2-2                         [1, 3]                    --
│    └─Linear: 2-3                       [1, 1]                    4
│    └─Sigmoid: 2-4                      [1, 1]                    --
Total params: 97
Trainable params: 97
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [61]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [66]:
for i in range(50):
  optimizer.zero_grad()
  y_pred = model(X_tr_train.to(dev))
  loss = criterion(y_pred.squeeze(), y_tr_train)
  print(f"Epoch: {i+1}  Loss: {loss.item():.6f}")
  loss.backward()
  optimizer.step()

Epoch: 1  Loss: 0.144922
Epoch: 2  Loss: 0.144247
Epoch: 3  Loss: 0.143573
Epoch: 4  Loss: 0.142907
Epoch: 5  Loss: 0.142246
Epoch: 6  Loss: 0.141593
Epoch: 7  Loss: 0.140949
Epoch: 8  Loss: 0.140306
Epoch: 9  Loss: 0.139667
Epoch: 10  Loss: 0.139031
Epoch: 11  Loss: 0.138402
Epoch: 12  Loss: 0.137782
Epoch: 13  Loss: 0.137169
Epoch: 14  Loss: 0.136552
Epoch: 15  Loss: 0.135945
Epoch: 16  Loss: 0.135339
Epoch: 17  Loss: 0.134737
Epoch: 18  Loss: 0.134141
Epoch: 19  Loss: 0.133547
Epoch: 20  Loss: 0.132951
Epoch: 21  Loss: 0.132363
Epoch: 22  Loss: 0.131779
Epoch: 23  Loss: 0.131191
Epoch: 24  Loss: 0.130609
Epoch: 25  Loss: 0.130044
Epoch: 26  Loss: 0.129481
Epoch: 27  Loss: 0.128912
Epoch: 28  Loss: 0.128342
Epoch: 29  Loss: 0.127786
Epoch: 30  Loss: 0.127237
Epoch: 31  Loss: 0.126691
Epoch: 32  Loss: 0.126146
Epoch: 33  Loss: 0.125601
Epoch: 34  Loss: 0.125124
Epoch: 35  Loss: 0.124628
Epoch: 36  Loss: 0.124124
Epoch: 37  Loss: 0.123604
Epoch: 38  Loss: 0.123069
Epoch: 39  Loss: 0.12

In [68]:
with torch.no_grad():
  y_t = model(X_tr_test)
  l = criterion(y_t.squeeze(), y_tr_test)
  print(f"loss: {l.item():.6f}")

loss: 0.109342
